# GeoThought Baseline Evaluation

Evaluates three autoregressive baselines on the **same validation split** used by LatentEuclid (`eval_e2e.py`),
so results are directly comparable to the reported 45.38% SOTA.

| # | Model | Input | Generation |
|---|-------|-------|------------|
| 1 | `Qwen3-VL-4B-Instruct` direct | image + question | greedy, 50 tokens |
| 2 | `Qwen3-4B-Base` text-only | question only (no image) | greedy, 50 tokens |
| 3 | `Qwen3-VL-4B-Instruct` reasoning | image + question | sampled, 1024 tokens |

**Parsing parity**: all three models go through the same `extract_and_clean()` pipeline
as `eval_e2e.py` — split on the last `Answer:` marker, then apply `clean_base_model_ans`.

**Output**: four JSON files saved to `RESULTS_DIR` on Drive:
- `eval_qwen_vl_direct.json`, `eval_qwen_base_text.json`, `eval_qwen_vl_reasoning.json`
- `eval_baseline_combined.json` — per-sample all-model view for offline cross-model analysis
- `eval_baseline_summary.json` — accuracy summary

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os

REPO_PATH   = "/content/drive/MyDrive/11-977/MMML-Project"
RESULTS_DIR = "/content/drive/MyDrive/11-977/checkpoints/baselines"
DATA_CACHE  = "/content/drive/MyDrive/11-977/MMML-Project/data/geothought_hf"

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Repo:     {REPO_PATH}")
print(f"Results:  {RESULTS_DIR}")


In [ ]:
# ── Cell 2: Device detection + dependency install ────────────────────────────
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "accelerate", "datasets", "pillow", "pyyaml"],
    check=False
)

import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"
    print("No GPU found — falling back to CPU")

import transformers as _tf
print(f"transformers: {_tf.__version__}  |  Device: {DEVICE}")


In [ ]:
# ── Cell 3: Dataset + split ───────────────────────────────────────────────────
# Split MUST match eval_e2e.py exactly (random.seed(42) + shuffle + last 10%).
import json, re, random
from PIL import Image
from datasets import load_dataset, load_from_disk

JSONL_PATH = os.path.join(REPO_PATH, "data/geothoughts_verified.jsonl")
GT_PATH    = os.path.join(REPO_PATH, "data/ground_truths.json")

if os.path.exists(DATA_CACHE):
    print("Loading HF dataset from Drive cache...")
    hf_dataset = load_from_disk(DATA_CACHE)
else:
    print("Downloading from HuggingFace (one-time)...")
    hf_dataset = load_dataset("xinlingdedeng/Geo-Thought", split="train")
    hf_dataset.save_to_disk(DATA_CACHE)
print(f"HF dataset size: {len(hf_dataset)}")

with open(JSONL_PATH) as f:
    full_data = [json.loads(line) for line in f]

with open(GT_PATH) as f:
    ground_truths = json.load(f)

# Identical split to eval_e2e.py
random.seed(42)
random.shuffle(full_data)
split_idx = int(0.9 * len(full_data))
val_data  = full_data[split_idx:]
print(f"Val split: {len(val_data)} samples (last 10% after seed-42 shuffle)")

def _load_image(item):
    m = re.search(r'sample_(\d+)', item.get("image_path", ""))
    if m:
        hf_row = hf_dataset[int(m.group(1))]
        img = hf_row.get("image") or hf_row.get("images")
        if img is not None:
            return img.convert("RGB") if hasattr(img, "convert") else Image.fromarray(img).convert("RGB")
    return Image.new("RGB", (224, 224), (128, 128, 128))

# Build val_items: drop samples with no ground truth (consistent with eval_e2e.py)
val_items = []
for item in val_data:
    gt = ground_truths.get(item["image_path"])
    if gt is None:
        continue
    val_items.append({
        "image_path": item["image_path"],
        "question":   item["question"],
        "gt_raw":     str(gt),
        "image":      None,
    })
print(f"Val items with ground truth: {len(val_items)}")

print("Pre-loading images from HF dataset...")
for item in val_items:
    item["image"] = _load_image(item)
print("Done.")


In [ ]:
# ── Cell 4: Evaluation utilities ──────────────────────────────────────────────
# Inlined from data/evaluate_generated.py to avoid PYTHONPATH dependencies.
# Functions are UNCHANGED — identical to what eval_e2e.py imports — so parsing
# is bit-for-bit equivalent across all baselines and LatentEuclid.
import math as _math
import re as _re

def clean_base_model_ans(ans):
    """From data/evaluate_generated.py — unchanged."""
    ans = ans.strip()
    ans = _re.sub(r'[*]*angstroms.*', '', ans, flags=_re.DOTALL)
    ans = _re.sub(r'[*]*wsj.*', '', ans, flags=_re.DOTALL)
    ans = _re.split(r'[\n\u4e00-\u9fff\u3040-\u30ff]', ans)[0]
    match = _re.match(r'^([-+]?[\d\./ \s*+\-()pisqrt]+)', ans)
    if match:
        ans = match.group(1)
    return ans.strip()

def safe_math_eval(expr_str):
    """From data/evaluate_generated.py — unchanged."""
    expr_str = _re.sub(r'(\d)(pi|sqrt)', r'\1*\2', str(expr_str))
    expr_str = _re.sub(r'\)(pi|sqrt|\d)', r')*\1', expr_str)
    expr_str = _re.sub(r'(pi)(\d)', r'\1*\2', expr_str)
    expr_str = expr_str.replace(')(', ')*(')     
    allowed_names = {"pi": _math.pi, "sqrt": _math.sqrt, "__builtins__": {}}
    safe_chars = set("0123456789.+-*/()pisqrt ")
    if not all(c in safe_chars for c in expr_str):
        return None
    import ast, warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            ast.parse(expr_str)
            val = eval(expr_str, allowed_names)
            return float(val) if val is not None else None
        except Exception:
            return None

def normalize(ans):
    """From data/evaluate_generated.py — unchanged."""
    ans = str(ans).strip()
    ans = _re.sub(r'(?:cm|mm|km|m|meters?|inches?|feet|foot|ft|nautical\smiles?|nauticalmiles?|nautical|miles?|yards?|yd|units?|square|sq|s|in)\b',
                  '', ans, flags=_re.IGNORECASE)
    ans = ans.replace('^2', '').replace('^', '').replace(' ', '')
    ans = ans.replace('(', '').replace(')', '').replace('[', '').replace(']', '')
    ans = ans.replace(':', '/')
    f_val = safe_math_eval(ans)
    if f_val is not None:
        return str(int(f_val)) if f_val == int(f_val) else str(round(f_val, 4))
    return ans.lower()

def extract_and_clean(generation: str) -> str:
    """
    Unified answer extractor used identically for all baselines.
    Mirrors y_decoder_prefix.py's generate() which splits on 'Answer:',
    then eval_e2e.py's clean_base_model_ans call.
    """
    if "Answer:" in generation:
        generation = generation.split("Answer:")[-1]
    return clean_base_model_ans(generation)

def score(pred_raw: str, gt_raw: str):
    """Returns (is_correct, gt_norm, pred_norm) — same logic as eval_e2e.py."""
    gt_norm   = normalize(gt_raw)
    pred_norm = normalize(pred_raw)
    is_correct = (gt_norm == pred_norm)
    if not is_correct:
        gt_val   = safe_math_eval(gt_raw)
        pred_val = safe_math_eval(pred_raw)
        if gt_val is not None and pred_val is not None:
            is_correct = _math.isclose(gt_val, pred_val, rel_tol=1e-3, abs_tol=0.06)
    return is_correct, gt_norm, pred_norm

print("Evaluation utilities ready.")

In [ ]:
# ── Cell 5: Shared evaluation runner ─────────────────────────────────────────
import gc
from tqdm import tqdm

# DEVICE is set in Cell 2 based on hardware detection.

def run_evaluation(val_items, generate_fn, model_name, batch_size, output_path):
    """
    Generic eval loop used by all three baselines.

    Args:
        val_items:    list of dicts with keys 'image', 'question', 'image_path', 'gt_raw'
        generate_fn:  callable(images: list[PIL], questions: list[str]) -> list[str]
                      Returns raw model generation strings (before answer extraction).
        model_name:   label for tqdm and summary output
        batch_size:   micro-batch size passed to generate_fn
        output_path:  where to save the per-sample result JSON

    Returns:
        results: list of per-sample dicts (same schema as eval_e2e.py outputs)
        summary: dict with 'correct', 'total', 'accuracy'
    """
    results = []
    correct = 0

    for i in tqdm(range(0, len(val_items), batch_size), desc=model_name):
        batch  = val_items[i : i + batch_size]
        images = [item["image"]    for item in batch]
        qs     = [item["question"] for item in batch]

        generations = generate_fn(images, qs)

        for item, gen in zip(batch, generations):
            pred_raw = extract_and_clean(gen)
            is_ok, gt_norm, pred_norm = score(pred_raw, item["gt_raw"])
            if is_ok:
                correct += 1
            results.append({
                "image":            item["image_path"],
                "is_correct":       is_ok,
                "gt_raw":           item["gt_raw"],
                "pred_raw":         pred_raw,
                "gt_norm":          gt_norm,
                "pred_norm":        pred_norm,
                "model_generation": gen.strip(),
            })

        # Periodic save so results survive a crash
        if (i // batch_size + 1) % 10 == 0:
            with open(output_path, "w") as f:
                json.dump(results, f, indent=2)

    total = len(results)
    acc   = correct / total * 100 if total > 0 else 0.0
    summary = {"model": model_name, "correct": correct, "total": total, "accuracy": acc}

    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"Correct: {correct}/{total}  |  Accuracy: {acc:.2f}%")
    print(f"Saved to {output_path}")
    print('='*50)
    return results, summary

print("Runner ready.")


## Models 1 & 3 — `Qwen3-VL-4B-Instruct`

Both variants share the same model weights; only the system prompt and generation parameters differ.
Load once, run both, then unload before loading the base model.

- **Direct**: concise system prompt, greedy decoding, 50 new tokens.
- **Reasoning**: step-by-step system prompt, sampled decoding (`temperature=0.7`), 1 024 new tokens.
  Sampling is necessary to activate the model's chain-of-thought mode.

Both use `<image>` stripped from the question text (the image is provided via the content array).

In [ ]:
# ── Cell 6: Load Qwen3-VL-4B-Instruct ────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText

VL_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

print(f"Loading {VL_MODEL_ID}...")

# Qwen3-VL requires transformers >= 4.52. If AutoProcessor returns a bare
# Qwen2TokenizerFast instead of a VL processor, images will be silently
# ignored and the VL baseline will be meaningless. Detect and warn.
vl_processor = AutoProcessor.from_pretrained(VL_MODEL_ID, trust_remote_code=True)

_proc_type = type(vl_processor).__name__
print(f"Processor type: {_proc_type}")

_is_vl_processor = hasattr(vl_processor, "image_processor")
if not _is_vl_processor:
    import transformers as _tf
    print(f"\n⚠  WARNING: AutoProcessor returned '{_proc_type}' (no image_processor).")
    print(f"   transformers version: {_tf.__version__}")
    print("   Qwen3-VL requires transformers >= 4.52.")
    print("   Run:  pip install -U transformers  then restart the kernel.")
    print("   Image inputs will NOT be encoded — VL results will be text-only.\n")

# AutoProcessor for Qwen3-VL may return a composite VL processor (with .tokenizer
# attribute) or a plain tokenizer directly (old transformers). Handle both.
_vl_tok = getattr(vl_processor, "tokenizer", vl_processor)
_vl_tok.padding_side = "left"   # required for batched left-padded generation
if _vl_tok.pad_token is None:
    _vl_tok.pad_token = _vl_tok.eos_token

vl_model = AutoModelForImageTextToText.from_pretrained(
    VL_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map={"" : DEVICE},
)
vl_model.eval()
params_b = sum(p.numel() for p in vl_model.parameters()) / 1e9
print(f"Model loaded ({params_b:.2f}B params). VL processor: {_is_vl_processor}")


In [ ]:
# ── Cell 7: Model 1 — VL direct (no reasoning, greedy) ───────────────────────
VL_DIRECT_BATCH   = 8
VL_DIRECT_MAX_TOK = 50
_SYSTEM_DIRECT = "You are a concise math assistant. Answer geometry questions with the numerical value only. No explanation, no units."

def _vl_build_inputs(images, questions, system_prompt):
    """Build batched processor inputs for Qwen3-VL-4B-Instruct."""
    messages_batch = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text":  q.replace("<image>", "").strip() + "\nAnswer:"},
            ]},
        ]
        for img, q in zip(images, questions)
    ]
    texts = [
        vl_processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    return vl_processor(
        text=texts, images=images, padding=True, return_tensors="pt"
    ).to(DEVICE)

def vl_direct_generate(images, questions):
    inputs = _vl_build_inputs(images, questions, _SYSTEM_DIRECT)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = vl_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=VL_DIRECT_MAX_TOK,
                repetition_penalty=1.05,
                pad_token_id=_vl_tok.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return _vl_tok.batch_decode(new_tokens, skip_special_tokens=True)

results_vl_direct, summary_vl_direct = run_evaluation(
    val_items,
    generate_fn=vl_direct_generate,
    model_name="Qwen3-VL-4B-Instruct (direct)",
    batch_size=VL_DIRECT_BATCH,
    output_path=os.path.join(RESULTS_DIR, "eval_qwen_vl_direct.json"),
)


In [ ]:
# ── Cell 8: Model 3 — VL reasoning (step-by-step, sampled) ───────────────────
VL_REASONING_BATCH   = 2
VL_REASONING_MAX_TOK = 1024
_SYSTEM_REASONING = (
    "You are a geometry expert. Think step by step before answering. "
    "Show your reasoning, then end with exactly: Answer: [number]"
)

def vl_reasoning_generate(images, questions):
    inputs = _vl_build_inputs(images, questions, _SYSTEM_REASONING)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = vl_model.generate(
                **inputs,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                max_new_tokens=VL_REASONING_MAX_TOK,
                repetition_penalty=1.05,
                pad_token_id=_vl_tok.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return _vl_tok.batch_decode(new_tokens, skip_special_tokens=True)

results_vl_reasoning, summary_vl_reasoning = run_evaluation(
    val_items,
    generate_fn=vl_reasoning_generate,
    model_name="Qwen3-VL-4B-Instruct (reasoning)",
    batch_size=VL_REASONING_BATCH,
    output_path=os.path.join(RESULTS_DIR, "eval_qwen_vl_reasoning.json"),
)


In [ ]:
# ── Cell 9: Unload VL model ───────────────────────────────────────────────────
del vl_model, vl_processor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free after VL model unload: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


## Model 2 — `Qwen3-4B-Base` text-only

Uses the **same base model** as the LatentEuclid Y-Decoder, with no image input.
This isolates the contribution of visual grounding: if LatentEuclid beats this baseline,
the latent thought vectors carry information the question text alone cannot provide.

Prompt format: `"{question}\nAnswer:"` — identical continuation style to the Y-Decoder's
`generate()` method (`text_prompts = [q.strip() + "\nAnswer: " for q in questions]`).
The `<image>` placeholder is stripped since no visual input is given.

In [ ]:
# ── Cell 10: Load Qwen3-4B-Base ───────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen3-4B-Base"

print(f"Loading {BASE_MODEL_ID} on {DEVICE}...")
print("First run will download ~8 GB — subsequent runs use the local HF cache.")

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_tokenizer.padding_side = "left"
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map={"": DEVICE},
)
base_model.eval()
print(f"Model loaded ({sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B params).")


In [ ]:
# ── Cell 11: Model 2 — Base text-only ────────────────────────────────────────
# Continuation format mirrors y_decoder_prefix.generate() exactly:
#   prompt = question.strip() + "\nAnswer: "
# repetition_penalty=1.1 prevents hallucination loops — same value as y_decoder_prefix.py.
BASE_BATCH   = 32
BASE_MAX_TOK = 50

def base_text_generate(images, questions):
    # images arg is ignored — text-only ablation
    prompts = [
        q.replace("<image>", "").strip() + "\nAnswer:"
        for q in questions
    ]
    inputs = base_tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(DEVICE)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = base_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=BASE_MAX_TOK,
                repetition_penalty=1.1,
                eos_token_id=[base_tokenizer.eos_token_id, 151645, 151643],
                pad_token_id=base_tokenizer.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return base_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

results_base_text, summary_base_text = run_evaluation(
    val_items,
    generate_fn=base_text_generate,
    model_name="Qwen3-4B-Base (text-only)",
    batch_size=BASE_BATCH,
    output_path=os.path.join(RESULTS_DIR, "eval_qwen_base_text.json"),
)


In [ ]:
# ── Cell 12: Unload base model ────────────────────────────────────────────────
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free after base model unload: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


## Results — Summary & Combined Output

In [ ]:
# ── Cell 13: Combine results + save ──────────────────────────────────────────
def _load_results(filename):
    path = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return []

def _index_by_image(results):
    return {r["image"]: r for r in results}

results_vl_direct    = _load_results("eval_qwen_vl_direct.json")
results_vl_reasoning = _load_results("eval_qwen_vl_reasoning.json")

idx_direct    = _index_by_image(results_vl_direct)
idx_base      = _index_by_image(results_base_text)
idx_reasoning = _index_by_image(results_vl_reasoning)

combined = []
for item in val_items:
    key = item["image_path"]
    def _entry(d):
        return {
            "model_generation": d["model_generation"],
            "pred_raw":         d["pred_raw"],
            "pred_norm":        d["pred_norm"],
            "is_correct":       d["is_correct"],
        }
    combined.append({
        "image":             key,
        "question":          item["question"],
        "gt_raw":            item["gt_raw"],
        "gt_norm":           normalize(item["gt_raw"]),
        "qwen_vl_direct":    _entry(idx_direct[key])    if key in idx_direct    else None,
        "qwen_base_text":    _entry(idx_base[key])      if key in idx_base      else None,
        "qwen_vl_reasoning": _entry(idx_reasoning[key]) if key in idx_reasoning else None,
    })

combined_path = os.path.join(RESULTS_DIR, "eval_baseline_combined.json")
with open(combined_path, "w") as f:
    json.dump(combined, f, indent=2)
print(f"Combined ({len(combined)} samples) saved to {combined_path}")

LATENT_EUCLID_V4 = {"model": "LatentEuclid v4 (SOTA)", "correct": None, "total": 595, "accuracy": 45.38}
summaries = [summary_base_text, LATENT_EUCLID_V4]
if results_vl_direct:
    vl_direct_correct = sum(1 for r in results_vl_direct if r["is_correct"])
    summaries.insert(0, {"model": "Qwen3-VL-4B-Instruct (direct)",
                         "correct": vl_direct_correct, "total": len(results_vl_direct),
                         "accuracy": vl_direct_correct / len(results_vl_direct) * 100})
if results_vl_reasoning:
    vl_reason_correct = sum(1 for r in results_vl_reasoning if r["is_correct"])
    summaries.insert(-1, {"model": "Qwen3-VL-4B-Instruct (reasoning)",
                           "correct": vl_reason_correct, "total": len(results_vl_reasoning),
                           "accuracy": vl_reason_correct / len(results_vl_reasoning) * 100})

summary_path = os.path.join(RESULTS_DIR, "eval_baseline_summary.json")
with open(summary_path, "w") as f:
    json.dump(summaries, f, indent=2)

print("\n" + "="*60)
print(f"{'Model':<42} {'Correct':>8} {'Total':>7} {'Acc':>8}")
print("-"*60)
for s in summaries:
    correct_str = str(s['correct']) if s['correct'] is not None else "—"
    print(f"{s['model']:<42} {correct_str:>8} {s['total']:>7} {s['accuracy']:>7.2f}%")
print("="*60)

_summary_map     = {s["model"]: s for s in summaries}
summary_vl_direct    = _summary_map.get("Qwen3-VL-4B-Instruct (direct)")
summary_vl_reasoning = _summary_map.get("Qwen3-VL-4B-Instruct (reasoning)")


In [ ]:
# ── Cell 14: Analysis — accuracy comparison bar chart ────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Build the display list dynamically — only include models that have results.
_all_models = [
    ("VL-4B\ndirect",        summary_vl_direct,    "#4C72B0"),
    ("Base-4B\ntext-only",   summary_base_text,    "#C44E52"),
    ("VL-4B\nreasoning",     summary_vl_reasoning, "#55A868"),
    ("LatentEuclid\nv4 SOTA", LATENT_EUCLID_V4,   "#DD8452"),
]
_plot_models = [(lbl, s, c) for lbl, s, c in _all_models if s is not None]

model_labels = [m[0] for m in _plot_models]
accs         = [m[1]["accuracy"] for m in _plot_models]
colors       = [m[2] for m in _plot_models]

# Cross-model agreement matrix — only for baselines with actual result lists
_result_lists = []
_result_labels = []
for key, lst, label in [
    ("Qwen3-VL-4B-Instruct (direct)",    results_vl_direct,    "VL direct"),
    ("Qwen3-4B-Base (text-only)",         results_base_text,    "Base text"),
    ("Qwen3-VL-4B-Instruct (reasoning)",  results_vl_reasoning, "VL reason"),
]:
    if lst:
        _result_lists.append({r["image"] for r in lst if r["is_correct"]})
        _result_labels.append(label)

n_result_sets = len(_result_lists)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: accuracy bar chart ──────────────────────────────────────────────────
bars = axes[0].bar(model_labels, accs, color=colors, edgecolor="white", linewidth=0.8)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{acc:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].set_ylim(0, 75)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("GeoThought Val Accuracy — Baselines vs LatentEuclid v4")
axes[0].axhline(25, color="gray", linestyle="--", linewidth=0.8, label="Random (25%)")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

# ── Right: cross-model agreement matrix ──────────────────────────────────────
if n_result_sets >= 2:
    all_images = {item["image_path"] for item in val_items}
    mat = np.zeros((n_result_sets, n_result_sets))
    for i in range(n_result_sets):
        for j in range(n_result_sets):
            mat[i, j] = len(_result_lists[i] & _result_lists[j]) / len(all_images) * 100

    im = axes[1].imshow(mat, cmap="Blues", vmin=0, vmax=100)
    axes[1].set_xticks(range(n_result_sets)); axes[1].set_xticklabels(_result_labels)
    axes[1].set_yticks(range(n_result_sets)); axes[1].set_yticklabels(_result_labels)
    axes[1].set_title("Cross-model correct-agreement (% of val set)")
    for i in range(n_result_sets):
        for j in range(n_result_sets):
            axes[1].text(j, i, f"{mat[i,j]:.1f}%", ha="center", va="center",
                         color="white" if mat[i,j] > 50 else "black", fontsize=9)
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
else:
    axes[1].text(0.5, 0.5, "Need ≥ 2 baselines\nfor agreement matrix",
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=12, color="gray")
    axes[1].set_title("Cross-model correct-agreement (% of val set)")
    axes[1].axis("off")

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "baseline_accuracy_comparison.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")


In [ ]:
# ── Cell 15: Analysis — error pattern breakdown ───────────────────────────────
# Classifies each incorrect prediction into:
#   'empty'   — model produced no parseable answer
#   'wrong'   — model produced a number but it was wrong
# This mirrors the Report 3 latent-to-language gap analysis.

def classify_errors(results):
    empty, wrong = 0, 0
    for r in results:
        if r["is_correct"]:
            continue
        if not r["pred_norm"] or r["pred_norm"] in ("", "none", "nan"):
            empty += 1
        else:
            wrong += 1
    return empty, wrong

# Only include baselines that were actually run this session or loaded from disk
model_data = []
for name, results, summary in [
    ("VL direct",    results_vl_direct,    summary_vl_direct),
    ("Base text",    results_base_text,    summary_base_text),
    ("VL reasoning", results_vl_reasoning, summary_vl_reasoning),
]:
    if results and summary is not None:
        model_data.append((name, results, summary))

if not model_data:
    print("No baseline results available yet. Run Cells 7, 11, or 8 first.")
else:
    fig, ax = plt.subplots(figsize=(max(6, 3 * len(model_data)), 4))

    x      = np.arange(len(model_data))
    labels = [d[0] for d in model_data]
    corr   = [d[2]["correct"] for d in model_data]
    empties, wrongs = [], []
    for _, results, _ in model_data:
        e, w = classify_errors(results)
        empties.append(e)
        wrongs.append(w)

    width = 0.25
    ax.bar(x - width, corr,    width, label="Correct",               color="#55A868")
    ax.bar(x,         wrongs,  width, label="Wrong answer",          color="#C44E52")
    ax.bar(x + width, empties, width, label="Empty / unparseable",   color="#8c8c8c")

    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Sample count")
    ax.set_title("Error breakdown per baseline model")
    ax.legend(); ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    err_fig_path = os.path.join(RESULTS_DIR, "baseline_error_breakdown.png")
    plt.savefig(err_fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("\nError summary:")
    print(f"{'Model':<18} {'Correct':>8} {'Wrong':>8} {'Empty':>8}")
    print("-" * 45)
    for (name, _, summ), e, w in zip(model_data, empties, wrongs):
        print(f"{name:<18} {summ['correct']:>8} {w:>8} {e:>8}")


In [ ]:
# ── Cell 16: Analysis — per-model unique contributions ───────────────────────
# Shows which examples ONLY one model gets right, useful for understanding
# complementary strengths. Only runs if at least 2 baselines have results.

import random as _rand

_avail = {
    "VL direct":    (results_vl_direct,    "qwen_vl_direct"),
    "Base text":    (results_base_text,    "qwen_base_text"),
    "VL reasoning": (results_vl_reasoning, "qwen_vl_reasoning"),
}
_avail = {k: v for k, v in _avail.items() if v[0]}   # drop empty

if len(_avail) < 2:
    print("Need ≥ 2 baselines for cross-model analysis. Run more cells first.")
else:
    all_img = {item["image_path"] for item in val_items}
    correct_sets = {k: {r["image"] for r in v[0] if r["is_correct"]} for k, v in _avail.items()}
    names = list(correct_sets.keys())

    print("Cross-model correctness overlap")
    if len(names) == 3:
        a, b, c = [correct_sets[n] for n in names]
        print(f"  All 3 correct:                  {len(a & b & c)}")
        for n, s in zip(names, [a, b, c]):
            others = [correct_sets[m] for m in names if m != n]
            unique = s - others[0] - others[1]
            print(f"  {n} only:                  {len(unique)}")
        print(f"  All 3 wrong:                    {len(all_img - a - b - c)}")
    else:
        # 2-model case
        n1, n2 = names
        s1, s2 = correct_sets[n1], correct_sets[n2]
        print(f"  Both correct:                   {len(s1 & s2)}")
        print(f"  {n1} only:             {len(s1 - s2)}")
        print(f"  {n2} only:             {len(s2 - s1)}")
        print(f"  Both wrong:                     {len(all_img - s1 - s2)}")

    # Sample examples each model uniquely gets right
    combined_by_img = {c["image"]: c for c in combined}
    _rand.seed(0)
    for label, (lst, combined_key) in _avail.items():
        others = [correct_sets[m] for m in names if m != label]
        unique = correct_sets[label]
        for o in others:
            unique = unique - o
        if not unique:
            continue
        sample = _rand.sample(sorted(unique), min(3, len(unique)))
        print(f"\n--- {label} only ({len(unique)} total) — up to 3 examples ---")
        for img in sample:
            c = combined_by_img.get(img, {})
            print(f"  Q:  {c.get('question','?')[:80]}")
            print(f"  GT: {c.get('gt_raw','?')}")
            entry = c.get(combined_key)
            if entry:
                print(f"  Pred: {entry['pred_raw']}  (✓={entry['is_correct']})")
            print()
